In [1]:
print("sql")

sql


In [ ]:
# 安装SQLmodel

from typing import Annotated
from fastapi import Depends,FastAPI,HTTPException,Query
from sqlmodel import Field,Session,SQLModel,create_engine,select


# class Hero(SQLModel,table =True):
#     __table_args__={"extend_existing":True}
#     id: int | None = Field(default=None,primary_key=True)
#     name:str = Field(index=True)
#     age:int | None =Field(default=None ,index=True)
#     secret_name :str

class HeroBase(SQLModel):
    name: str = Field(index=True)
    age: int | None = Field(default=None,index=True)


class Hero(HerBase,table=True):
    __table_args__ = {"extend_existing":True}
    id:int | None =Field(default=None,primary_key=True)
    secret_name: str

class HeroPublic(HeroBase):
    id: int

class HeroCreate(HeroBase):
    secret_name: str


sqlite_file_name = "database.db"
sqlite_url=f"sqlite:///{sqlite_file_name}"

connect_args = {"check_same_thread":False}
engine = create_engine(sqlite_url,connect_args=connect_args)


d:\code\vs\1\fast-api-demo\.venv\lib\site-packages\sqlmodel\main.py:681: SAWarning: This declarative base already contains a class with the same class name and module name as __main__.Hero, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


In [11]:
# 为所有表模型创建表
def create_db_and_tables():
    SQLModel.metadata.create_all(engine)


def get_session():
    with Session(engine) as sessio:
        yield session

SessionDep = Annotated[Session,Depends(get_session)]


# 启动时创建表
@app.on_event("startup")
def  on_startup():
    create_db_and_tables()


NameError: name 'app' is not defined

In [13]:
app = FastAPI()
@app.post("/heroes/")
def create_hero(hero: Hero,session:SessionDep) ->Hero:
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero

In [14]:
@app.get("/heroes")
def read_heroes(session: SessionDep,offset: int =0,
                limit: Annotated[int,Query(le=100)] =100,) -> list[Hero]:
    heroes = session.exec(select(Hero).offset(offset).limit(limit))
    return heroes

In [15]:
@app.get("/heroes/{heroes_id}")
def read_heroe(hero_id: int ,session:SessionDep) ->Hero:
    hero  =session.get(Hero,hero_id)
    if not hero:
        raise HTTPException(status_code=404,detail="Hero not found")
    return hero


In [16]:
@app.delete("/heroes/{hero_id}")
def delete_hero(hero_id: int ,session: SessionDep):
    hero = session.get(Hero,hero_id)
    if not hero:
        raise HTTPException(status_code=404,detail="Hero not found")
    session.delete(hero)
    session.commit()
    return {"ok",True}
    

NameError: name 'HerBase' is not defined